<a href="https://colab.research.google.com/github/hannahandkush/Coursework/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In Google Colab or some other platform, prompt the AI bot with "Create a PyTorch script to fine-tune a pre-trained convolutional neural network for the MNIST data set". Fine tune for a couple of epochs and save the weights as a 'pth' file.

Deploy that trained CNN model trained on your Hugging Face space with Gradio. The Gradio interface should include some examples of input images and should output the estimated probability for each predicted digit.  You can try to add a  virtual pad to the interface where you can draw a number 0-9 to be classified by the predictive model.

You need to submit:

    The URL of your notebook in your GitHub repository to train the model. The pipeline should be similar the one discussed in class. The script should contain an instruction to save the model to be deployed. One should be able to execute the code on Colab just by clicking the "Open in Colab" button and executing the notebook.
    A link to your Hugging Face space where your model is deployed. The app (that should allow to either take a picture, load an image of a hand-written digit, or draw a digit directly using the app) should be running so it can be tested on the HF space;
    A video (3’ maximum) explaining how the model was saved and loaded, and how Gradio and Hugging Face spaces were used for deployment.


## Fine-tuning a Pre-trained CNN for MNIST

This notebook will demonstrate how to fine-tune a pre-trained Convolutional Neural Network (CNN) for the MNIST dataset. We will:

1.  Load and preprocess the MNIST dataset.
2.  Load a pre-trained CNN model (e.g., ResNet).
3.  Modify the last layer of the CNN to classify 10 digits.
4.  Define a training loop and train the model for a couple of epochs.
5.  Save the trained model's weights.

In [25]:
# 1. Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 2. Load and Preprocess the MNIST Dataset

MNIST images are grayscale (1 channel), but many pre-trained models expect 3-channel RGB images. We'll convert them to 3 channels by repeating the single channel three times. We'll also resize them to the input size expected by the pre-trained model (e.g., 224x224 for ResNet).

In [26]:
# Define transformations for the MNIST dataset
import torchvision.transforms as transforms # Added this import to resolve NameError
from torchvision import datasets # Added this import to resolve NameError

# Pre-trained models usually expect 224x224 input and normalization

# Training transformations with data augmentation
train_transform = transforms.Compose([
    transforms.Resize((28, 28)), # Corrected Resize to 28x28 for for MNIST size
    transforms.RandomRotation(degrees=15), # Random rotation up to 15 degrees
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Slight affine translation
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), # Convert 1 channel to 3 channels
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081)) # Normalize for 3 channels
])

# Test transformations (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((28, 28)), # Matches your app layout resizing
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), # Convert 1 channel to 3 channels
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081)) # Normalize for 3 channels
])

# Load MNIST training and test datasets
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

Number of training samples: 60000
Number of test samples: 10000


### 3. Load a Pre-trained CNN Model and Modify the Classifier

We'll use a pre-trained ResNet-18 model from `torchvision.models`. We'll freeze all layers except the final classification layer and then replace the final layer to match the 10 classes of MNIST.

In [27]:
# Load a pre-trained ResNet-18 model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all parameters in the network
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
# ResNet's final layer is `fc`
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10) # MNIST has 10 classes (0-9)

model = model.to(device)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

The `models.ResNet18_Weights.IMAGENET1K_V1` object contains information about the transformations used during the original training of ResNet-18 on ImageNet. You can inspect its `transforms()` method to see the expected input size and other preprocessing steps.

In [28]:
import torchvision.models as models

# Get the weights object
weights = models.ResNet18_Weights.IMAGENET1K_V1

# Get the transformations associated with these weights
transforms_config = weights.transforms()

print("Transforms associated with ResNet18_Weights.IMAGENET1K_V1:\n")
print(transforms_config)

# The default input size for ResNet models is typically 224x224,
# but you can often find it explicitly in the transforms or documentation.
# For torchvision models, the default `Resize` operation targets 256 then `CenterCrop` to 224.
# However, in your preprocessing, you directly resize to (224, 224).

# You can often infer the expected size from the `Resize` or `CenterCrop` transformations.
# In your current setup, you've explicitly resized to (224, 224) in your DataLoader transformations.


Transforms associated with ResNet18_Weights.IMAGENET1K_V1:

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


### 4. Define Loss Function, Optimizer, and Training Loop

We'll use Cross-Entropy Loss and an Adam optimizer. The training loop will iterate for a specified number of epochs.

In [29]:
# Evaluation function
def evaluate_model(model, test_loader, device):
    model.eval() # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad(): # No need to calculate gradients during evaluation
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    accuracy = correct / total
    return accuracy


In [30]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # Only optimize the new fc layer

# Training function modified to train until target accuracy is reached
def train_model(model, train_loader, criterion, optimizer, test_loader, evaluate_func, device, target_accuracy=0.70, max_epochs=10):
    model.train() # Set the model to training mode
    current_epoch = 0
    print(f"Training for a maximum of {max_epochs} epochs or until {100 * target_accuracy:.2f}% test accuracy.")
    while current_epoch < max_epochs:
        current_epoch += 1
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad() # Zero the parameter gradients

            outputs = model(data) # Forward pass
            loss = criterion(outputs, target) # Calculate loss
            loss.backward() # Backward pass
            optimizer.step() # Optimize

            running_loss += loss.item() * data.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += target.size(0)
            correct_predictions += (predicted == target).sum().item()

            if (batch_idx + 1) % 100 == 0: # Print every 100 batches
                print(f'Epoch [{current_epoch}], Batch [{batch_idx+1}/{len(train_loader)}], Train Loss: {loss.item():.4f}')

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_predictions / total_samples
        print(f'Epoch {current_epoch} finished. Avg Train Loss: {epoch_loss:.4f}, Train Accuracy: {100 * epoch_acc:.2f}%')

        # Evaluate on the test set after each epoch
        test_accuracy = evaluate_func(model, test_loader, device)
        print(f'Test Accuracy after Epoch {current_epoch}: {100 * test_accuracy:.2f}%')

        if test_accuracy >= target_accuracy:
            print(f"Target accuracy of {100 * target_accuracy:.2f}% reached. Stopping training.")
            break
    return test_accuracy

# Train the model until 98% accuracy on the test set (or max_epochs)
print("Starting training...")
final_test_accuracy = train_model(model, train_loader, criterion, optimizer, test_loader, evaluate_model, device, target_accuracy=0.98, max_epochs=15)
print(f"Training complete! Final Test Accuracy: {100 * final_test_accuracy:.2f}%")

# Save the model's state dictionary after training
model_save_path = 'mnist_cnn_finetuned.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

print("Done.")


Starting training...
Training for a maximum of 15 epochs or until 98.00% test accuracy.
Epoch [1], Batch [100/938], Train Loss: 1.3946
Epoch [1], Batch [200/938], Train Loss: 1.1880
Epoch [1], Batch [300/938], Train Loss: 1.3844
Epoch [1], Batch [400/938], Train Loss: 1.2429
Epoch [1], Batch [500/938], Train Loss: 1.2808
Epoch [1], Batch [600/938], Train Loss: 1.3972
Epoch [1], Batch [700/938], Train Loss: 1.2644
Epoch [1], Batch [800/938], Train Loss: 1.3308
Epoch [1], Batch [900/938], Train Loss: 1.1322
Epoch 1 finished. Avg Train Loss: 1.3178, Train Accuracy: 56.44%
Test Accuracy after Epoch 1: 71.17%
Epoch [2], Batch [100/938], Train Loss: 1.0676
Epoch [2], Batch [200/938], Train Loss: 1.0668
Epoch [2], Batch [300/938], Train Loss: 1.1872
Epoch [2], Batch [400/938], Train Loss: 1.1032
Epoch [2], Batch [500/938], Train Loss: 1.1800
Epoch [2], Batch [600/938], Train Loss: 0.9403
Epoch [2], Batch [700/938], Train Loss: 0.9795
Epoch [2], Batch [800/938], Train Loss: 1.1169
Epoch [2], B

### 5. Test Data Set

Sample images are extracted for each number to test tin the Gradio interface

In [31]:
from torchvision.datasets import MNIST
from PIL import Image
import os

os.makedirs("examples", exist_ok=True)
ds = MNIST(root=".", train=False, download=True)
found = {}
for img, label in ds:
    if label not in found:
        # Resize to 28x28 as MNIST images are typically 28x28
        img.resize((28, 28), Image.NEAREST).save(f"examples/digit_{label}.png")
        found[label] = True
    if len(found) == 10:
        break

## Deploying to Hugging Face Spaces

To deploy your model, you'll need two main things:

1.  **`requirements.txt`**: A file listing all Python dependencies your Gradio app will need.
2.  **`model_weights.pth`**: Your trained model weights.

Here's how to create these and push them to your Hugging Face Space.

In [32]:
requirements_content = """
torch
torchvision
numpy
pillow
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("requirements.txt created successfully.")

requirements.txt created successfully.


### 2. Install and Authenticate with `huggingface_hub`

We'll use the `huggingface_hub` library to interact with your Hugging Face Space. You'll need an API token, which you can get from your Hugging Face settings (Settings -> Access Tokens). It's best to store this as a Colab secret.

In [33]:
# Install huggingface_hub
!pip install huggingface_hub -q

To authenticate, you'll need a Hugging Face API token. Please get your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and add it as a Colab secret named `HF_TOKEN`.

Then run the cell below to log in.

In [34]:
# Authenticate with Hugging Face
from huggingface_hub import HfApi, notebook_login
from google.colab import userdata

# It's generally recommended to call notebook_login() without arguments
# as it automatically checks for the 'HF_TOKEN' secret in Colab.
print("Attempting to log in to Hugging Face...")
try:
    notebook_login()
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")
    print("Please ensure you have a secret named 'HF_TOKEN' and it's correctly configured, or log in manually with your token if not using secrets.")

Attempting to log in to Hugging Face...


### 3. Upload Files to Your Hugging Face Space

Now we can use the `HfApi` to upload your `mnist_cnn_finetuned.pth` file (you can rename it to `model_weights.pth` locally before uploading if you wish) and the generated `requirements.txt` to your specified Hugging Face Space. The target Hugging Face Space is `HannahandKush/ML`.

In [35]:
!cp /content/mnist_cnn_finetuned.pth . 2>/dev/null || cp ./ML/mnist_cnn_finetuned.pth . 2>/dev/null || find /content/ -name "mnist_cnn_finetuned.pth" -exec cp {} . \;

cp: '/content/mnist_cnn_finetuned.pth' and './mnist_cnn_finetuned.pth' are the same file


In [36]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "HannahandKush/ML"

print("Initiating secure upload stream to Hugging Face...")

try:
    # We use upload_folder or explicit individual files wrapped in a try block
    # to catch any hidden network rejections immediately.

    # 1. Upload the weights file
    api.upload_file(
        path_or_fileobj="mnist_cnn_finetuned.pth",
        path_in_repo="mnist_cnn_finetuned.pth",
        repo_id=repo_id,
        repo_type="space",
        commit_message="Upgraded model weights via Colab training loop"
    )
    print("✅ Model weights (.pth) uploaded and pushed to Space successfully!")

    # 2. Upload the requirements file
    api.upload_file(
        path_or_fileobj="requirements.txt",
        path_in_repo="requirements.txt",
        repo_id=repo_id,
        repo_type="space",
        commit_message="Updated requirements.txt dependencies"
    )
    print("✅ Requirements file uploaded successfully!")
    print(f"Check your updated space live history at: https://huggingface.co/spaces/{repo_id}/tree/main")

except Exception as e:
    print("\n❌ UPLOAD FAILED! The server rejected the file stream.")
    print(f"Error Details: {e}")
    print("\n👉 Troubleshooting Step: Go to hf.co/settings/tokens. Make sure the token you are using has 'WRITE' permissions enabled, not just 'READ'!")

Initiating secure upload stream to Hugging Face...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  mnist_cnn_finetuned.pth     :  97%|#########7| 43.6MB / 44.8MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model weights (.pth) uploaded and pushed to Space successfully!
✅ Requirements file uploaded successfully!
Check your updated space live history at: https://huggingface.co/spaces/HannahandKush/ML/tree/main


You can now navigate to your Hugging Face Space and ensure the files `model_weights.pth` and `requirements.txt` are present. You will then need to create your Gradio application file (e.g., `app.py`) in your Space to use these files.

### 4. Prepare and Upload Example Images to Hugging Face Space

To make your Gradio app more user-friendly, it's good practice to provide example images. We'll generate a subset of 30 random images from the MNIST dataset and upload them to an `hf_examples` folder in your Hugging Face Space.

In [37]:
import random
from PIL import Image
import numpy as np

# Create a directory for Hugging Face examples
hf_examples_dir = "hf_examples"
os.makedirs(hf_examples_dir, exist_ok=True)

# Get 30 random indices from the test dataset
num_samples = 30
random_indices = random.sample(range(len(test_dataset)), num_samples)

print(f"Saving {num_samples} random MNIST images to '{hf_examples_dir}'...")

for i, idx in enumerate(random_indices):
    img, label = test_dataset[idx]
    # MNIST images are 28x28 grayscale, convert to PIL Image if not already
    if isinstance(img, torch.Tensor):
        img = transforms.ToPILImage()(img)
    elif not isinstance(img, Image.Image):
        # Assuming it's a numpy array if not tensor or PIL Image
        img = Image.fromarray(img.numpy(), mode='L')

    # Save the image as a PNG
    img.save(os.path.join(hf_examples_dir, f"random_digit_{label}_{i}.png"))

print(f"Successfully saved {num_samples} example images to '{hf_examples_dir}'.")

Saving 30 random MNIST images to 'hf_examples'...
Successfully saved 30 example images to 'hf_examples'.


In [38]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "HannahandKush/ML" # Your Hugging Face Space ID
hf_examples_dir = "hf_examples"

print(f"Uploading folder '{hf_examples_dir}' to Hugging Face Space '{repo_id}'...")

try:
    api.upload_folder(
        folder_path=hf_examples_dir,
        repo_id=repo_id,
        repo_type="space",
        path_in_repo="examples", # Upload to an 'examples' folder in the HF Space
        commit_message="Added 30 random MNIST example images"
    )
    print(f"✅ Folder '{hf_examples_dir}' uploaded as 'examples' to Space successfully!")
    print(f"Check your updated space live history at: https://huggingface.co/spaces/{repo_id}/tree/main")

except Exception as e:
    print("\n❌ UPLOAD FAILED! The server rejected the folder stream.")
    print(f"Error Details: {e}")
    print("\n👉 Troubleshooting Step: Go to hf.co/settings/tokens. Make sure the token you are using has 'WRITE' permissions enabled, not just 'READ'!")

Uploading folder 'hf_examples' to Hugging Face Space 'HannahandKush/ML'...
✅ Folder 'hf_examples' uploaded as 'examples' to Space successfully!
Check your updated space live history at: https://huggingface.co/spaces/HannahandKush/ML/tree/main
